# Task 2 — Qiskit Circuits on Quokka (LO1, LO2)

This notebook contains:
1) The required Qiskit circuit (H on q0, X on q1, then a controlled gate C ≈ CX).
2) Execution on Quokka QASM backend and measurement results.
3) A custom circuit written as OpenQASM and executed on Quokka.

In [1]:
!pip -q install qiskit matplotlib pylatexenc requests

import json
import requests
from collections import Counter
import os

import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit import qasm2

os.makedirs("screenshots", exist_ok=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.6 MB/s eta 0:00:00


In [2]:
QUOKKA_URL = "http://quokka2.quokkacomputing.com/qsim/qasm"

def run_on_quokka(qasm_script: str, shots: int = 1024, url: str = QUOKKA_URL):
    payload = {"count": shots, "script": qasm_script}
    headers = {"Content-Type": "application/json"}
    r = requests.post(url, data=json.dumps(payload), headers=headers, timeout=60)
    r.raise_for_status()
    return r.json()

def extract_quokka_counts(resp: dict):
    """
    Typical Quokka response:
    {'error': 'no error', 'error_code': 0, 'result': {'c': [[0,0],[1,1],...]}}

    Returns:
      counts: Counter({'00': x, ...})
      creg_name: e.g. 'c'
    """
    if not isinstance(resp, dict):
        raise ValueError(f"Unexpected response type: {type(resp)}")

    if "result" not in resp or not isinstance(resp["result"], dict):
        raise ValueError(f"Could not find resp['result']. Response preview: {str(resp)[:400]}")

    creg_name = next(iter(resp["result"].keys()))
    data = resp["result"][creg_name]

    # list of lists -> bitstrings
    if isinstance(data, list) and len(data) > 0 and isinstance(data[0], list):
        bitstrings = ["".join(str(b) for b in row) for row in data]
        return Counter(bitstrings), creg_name

    # already bitstrings
    if isinstance(data, list) and len(data) > 0 and isinstance(data[0], str):
        return Counter(data), creg_name

    raise ValueError(f"Unexpected result format under key '{creg_name}': {type(data)}")

def counts_to_probs(counts: Counter):
    total = sum(counts.values())
    return {k: v/total for k, v in sorted(counts.items())}

In [3]:
qc_req = QuantumCircuit(2, 2)

# Required circuit (from assignment image):
qc_req.h(0)
qc_req.x(1)
qc_req.cx(0, 1)
qc_req.measure([0, 1], [0, 1])

qc_req

In [4]:
fig = qc_req.draw("mpl")
fig.savefig("screenshots/task2_required_circuit.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved: screenshots/task2_required_circuit.png")

Saved: screenshots/task2_required_circuit.png


In [5]:
shots = 1024
qasm_req = qasm2.dumps(qc_req)

print("QASM preview (required circuit):")
print(qasm_req)

resp_req = run_on_quokka(qasm_req, shots=shots)
print("\nQuokka error:", resp_req.get("error"), "| error_code:", resp_req.get("error_code"))

counts_req, creg_used_req = extract_quokka_counts(resp_req)
print("Detected classical register:", creg_used_req)
print("Quokka counts (required circuit):", counts_req)

print("\nProbabilities (required circuit):", counts_to_probs(counts_req))

QASM preview (required circuit):
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
h q[0];
x q[1];
cx q[0],q[1];
measure q[0] -> c[0];
measure q[1] -> c[1];

Quokka error: no error | error_code: 0
Detected classical register: c
Quokka counts (required circuit): Counter({'01': 545, '10': 479})

Probabilities (required circuit): {'01': 0.5322265625, '10': 0.4677734375}


In [6]:
fig = plot_histogram(dict(counts_req))
fig.savefig("screenshots/task2_required_hist.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved: screenshots/task2_required_hist.png")

Saved: screenshots/task2_required_hist.png


In [7]:
qc_custom = QuantumCircuit(3, 3)

# Custom circuit: mix of single-qubit phase gates + entangling gates + Toffoli
qc_custom.h(0)
qc_custom.s(0)
qc_custom.cx(0, 1)
qc_custom.t(1)
qc_custom.x(2)
qc_custom.ccx(0, 1, 2)  # Toffoli
qc_custom.barrier()
qc_custom.measure([0, 1, 2], [0, 1, 2])

qc_custom

In [8]:
fig = qc_custom.draw("mpl")
fig.savefig("screenshots/task2_custom_circuit.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved: screenshots/task2_custom_circuit.png")

Saved: screenshots/task2_custom_circuit.png


In [9]:
qasm_custom = qasm2.dumps(qc_custom)

print("QASM (custom circuit):")
print(qasm_custom)

resp_custom = run_on_quokka(qasm_custom, shots=shots)
print("\nQuokka error:", resp_custom.get("error"), "| error_code:", resp_custom.get("error_code"))

counts_custom, creg_used_custom = extract_quokka_counts(resp_custom)
print("Detected classical register:", creg_used_custom)
print("Quokka counts (custom circuit):", counts_custom)

print("\nProbabilities (custom circuit):", counts_to_probs(counts_custom))

QASM (custom circuit):
OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
creg c[3];
h q[0];
s q[0];
cx q[0],q[1];
t q[1];
x q[2];
ccx q[0],q[1],q[2];
barrier q[0],q[1],q[2];
measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[2] -> c[2];

Quokka error: no error | error_code: 0
Detected classical register: c
Quokka counts (custom circuit): Counter({'110': 515, '001': 509})

Probabilities (custom circuit): {'001': 0.4970703125, '110': 0.5029296875}


In [10]:
fig = plot_histogram(dict(counts_custom))
fig.savefig("screenshots/task2_custom_hist.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved: screenshots/task2_custom_hist.png")

Saved: screenshots/task2_custom_hist.png
